# Bengaluru House Price Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost · TabPFN-v2
Baselines: LazyPredict · FLAML AutoML
Dataset: 13,320 rows × 9 columns
Target: `price`

## 2 · Project Overview

We predict **house prices in Bengaluru, India** based on area type, location, size, total square footage, number of bathrooms, and balconies. This is a medium-sized dataset (13,320 rows) with several preprocessing challenges: mixed-format `total_sqft` column, high-cardinality `location`, and properties with sizes in BHK format.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Handle missing values, categorical encoding, and feature engineering.
3. Build a baseline model and compare against modern boosting/AutoML.
4. Use LazyPredict for rapid benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Evaluate with RMSE, MAE, R², and residual analysis.
7. Select and analyze the top 3 models.

## 4 · Problem Statement

Predict the **price** (in lakhs INR) of a property in Bengaluru given its location, size, area type, square footage, bathrooms, and balconies.

## 5 · Why This Project Matters

- Real estate price prediction is a high-value business problem.
- The dataset has realistic messiness: mixed units, missing values, high-cardinality location.
- Cleaning this data teaches practical data wrangling skills.
- Bengaluru is one of India's fastest-growing cities with a dynamic housing market.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 13,320 |
| **Columns** | 9 |
| **Target** | price (lakhs INR) |
| **High-cardinality** | location (~1,300 unique values) |
| **Mixed format** | total_sqft (e.g., '1200', '1100 - 1400', '34.46Sq. Meter') |
| **Missing** | society (~42%), bath, balcony, size |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle / educational datasets.
- **License**: Public / educational use.
- **File**: `Bengaluru_House_Data.csv` (included locally).

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
import re
warnings.filterwarnings("ignore")
%matplotlib inline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "price"
SAVE_DIR = "."
print(f"Target: {TARGET}")

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = "Bengaluru_House_Data.csv"
_candidates = [DATA_PATH, "Regression/Bengaluru House Price Prediction/Bengaluru_House_Data.csv"]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), _candidates[0])
SAVE_DIR = os.path.dirname(os.path.abspath(DATA_PATH)) if os.path.exists(DATA_PATH) else "."

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

## 12 · Data Validation Checks

Check the messy `total_sqft` column, missing values in `society`, `bath`, and `balcony`.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

Explore price distributions across locations, area types, and property sizes.

In [ ]:
df.describe()

In [ ]:
# Correlation heatmap
num_cols = df.select_dtypes(include="number").columns.tolist()
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(min(len(num_cols)+2, 14), min(len(num_cols), 12)))
    sns.heatmap(corr, annot=len(num_cols)<=15, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
    ax.set_title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()

In [ ]:
# Feature distributions
num_plot = [c for c in num_cols if c != TARGET][:12]
if num_plot:
    n_r = (len(num_plot) + 3) // 4
    fig, axes = plt.subplots(n_r, min(4, len(num_plot)), figsize=(16, 3*n_r), squeeze=False)
    for i, col in enumerate(num_plot):
        r, c = divmod(i, 4)
        df[col].hist(bins=30, ax=axes[r][c], color="steelblue", edgecolor="black")
        axes[r][c].set_title(col, fontsize=9)
    for i in range(len(num_plot), n_r*min(4,len(num_plot))):
        r, c = divmod(i, 4)
        axes[r][c].set_visible(False)
    plt.suptitle("Feature Distributions")
    plt.tight_layout()
    plt.show()

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel(TARGET)
axes[1].boxplot(df[TARGET].dropna(), vert=True)
axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout()
plt.show()

print(f"Range: {df[TARGET].min():.2f} to {df[TARGET].max():.2f}")
print(f"Mean: {df[TARGET].mean():.2f}, Median: {df[TARGET].median():.2f}")
print(f"Std: {df[TARGET].std():.2f}, Skew: {df[TARGET].skew():.3f}")

## 15 · Train/Validation/Test Split Strategy

We use an **80/20 train/test split** with stratification where appropriate.

In [ ]:
df = df.dropna(subset=[TARGET]).copy()
y = df[TARGET]
X = df.drop(columns=[TARGET])

# Encode categoricals
cat_cols = X.select_dtypes(include=["object","category"]).columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

# Fill missing
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
for c in cat_cols:
    X[c] = X[c].fillna("unknown")

oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
if cat_cols:
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Sanitize column names
X.columns = [str(c).replace(" ","_").replace("&","_and_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **total_sqft**: Contains ranges ('1100 - 1400') and unit variations — parse to numeric average.
- **size**: Extract BHK count as integer.
- **society**: ~42% missing — drop this column.
- **location**: ~1,300 unique — group rare locations into 'other'.
- **bath/balcony**: Fill missing with median.

## 17 · Feature Engineering

Key preprocessing:
- Parse `total_sqft` ranges to averages.
- Extract BHK count from `size`.
- Group rare locations (< 10 occurrences) as 'other'.
- Drop `society` column (too many missing values).

These steps are done before the split in the loading phase.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()
print(f"Features: {X_train.columns.tolist()}")

## 18 · Baseline Model

Linear Regression as baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Rapid model screening across dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
_max = 5000
_Xtr = X_train.iloc[:_max] if len(X_train) > _max else X_train
_Xte = X_test.iloc[:_max] if len(X_test) > _max else X_test
_ytr = y_train.iloc[:_max] if len(y_train) > _max else y_train
_yte = y_test.iloc[:_max] if len(y_test) > _max else y_test
lazy_models, _ = lazy.fit(_Xtr, _Xte, _ytr, _yte)

print("LazyPredict — Top 10:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection (60s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60, verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}, R2: {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML failed: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=8,
                           task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                           device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8,
                             device="cuda", tree_method="hist", verbosity=0,
                             n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, min(3, len(top3_names)), figsize=(6*min(3,len(top3_names)), 5))
if len(top3_names) == 1:
    axes = [axes]

for i, name in enumerate(top3_names[:3]):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.4, s=15, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.0f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual"); axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names[:3]:
    yp = results[name]
    print(f"\n  {name}: RMSE={root_mean_squared_error(y_test,yp):,.2f} MAE={mean_absolute_error(y_test,yp):,.2f} R2={r2_score(y_test,yp):.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(residuals, bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual")

axes[1].scatter(best_pred, residuals, alpha=0.4, s=15)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].plot(np.sort(residuals), marker="o", ms=3, linestyle="-", color="steelblue")
axes[2].axhline(0, color="red", linestyle="--")
axes[2].set_title("Sorted Residuals")
plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Total square footage** and **BHK count** are the strongest predictors.
- **Location** matters significantly — premium areas command higher prices.
- **Area type** (Super built-up vs. Built-up vs. Plot) influences pricing.

**Business takeaway:** For real estate valuation, square footage and location are the primary drivers. The model can help identify under/over-priced properties.

## 26 · Limitations

1. High location cardinality requires grouping — loses granularity.
2. Price in lakhs without adjusting for market timing.
3. Missing society data (dropped) could be informative.
4. No temporal dimension — prices change over time.
5. Outliers in price and sqft can distort models.

## 27 · How to Improve This Project

1. Use target encoding for location instead of ordinal.
2. Add price per sqft as a feature.
3. Remove outlier properties (e.g., 2BHK with 10 bathrooms).
4. Add external data: proximity to metro, schools, hospitals.
5. Try log-transforming the target for better distribution.

## 28 · Production Considerations

- Wrap preprocessing in a reproducible pipeline.
- Output prediction intervals, not just point estimates.
- Monitor for data drift and retrain periodically.
- Validate new data against the training schema before prediction.

## 29 · Common Mistakes

1. Not handling missing values before model training.
2. Leaking test data into preprocessing (fit on train only).
3. Ignoring the baseline — if LR is close to tree models, the data may be mostly linear.
4. Over-tuning on a small test set — use cross-validation for robust estimates.
5. Not checking residual patterns for systematic bias.

## 30 · Mini Challenge / Exercises

1. Try removing the weakest features — does R² improve?
2. Compare Ridge/Lasso to see which features get zeroed out.
3. Use 5-fold cross-validation instead of a single train/test split.
4. Try log-transforming the target if it is skewed.
5. Add polynomial interaction features and see if tree models still win.

## 31 · Final Summary / Key Takeaways

1. Modern gradient boosting (CatBoost, LightGBM, XGBoost) typically dominates tabular regression.
2. LazyPredict + FLAML give rapid screening — use them to shortlist candidates.
3. Always compare against a simple baseline.
4. Residual analysis reveals systematic errors that metrics alone miss.
5. Feature engineering and proper preprocessing often matter more than model choice.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))